# 9. Predict

**Project:** Predicting colorectal cancer screening non-compliance to guide targeted outreach
**Part of:** the final deliverable -- takes new respondent-level input values (the same 17 M3
predictors used throughout `4a`-`8`) and returns a calibrated non-compliance risk score, using the
project's official model: `gb_M3_tuned_calibrated` (gradient boosting, tuned, isotonic-calibrated
in `5 Evaluation.ipynb`). This is the model to use here, not the two-stage models in
`8 Two-Stage Model.ipynb` -- see that notebook's closing note for why the single-stage calibrated
model remains the one carried forward.

**What this notebook provides:**
1. A reference list of every predictor's valid category values (pulled live from the training
   data, not hand-typed, so it can't drift out of sync).
2. A `predict_respondents()` function that validates input, scores it through the calibrated
   pipeline, and places the result in context against the test-set risk distribution (which
   decile it would fall in, consistent with `5 Evaluation.ipynb`'s risk-decile table).
3. Worked examples, plus a batch mode for scoring a CSV of new respondents at once.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)

sys.path.insert(0, str(Path.cwd()))
from pipeline_components import (
    M3_FEATURES, ORDINAL_FEATURES, load_prediction_context, predict_respondents,
    predict_across_models, COMPARISON_MODELS, MODEL_DISPLAY_NAMES,
)

DATA_DIR = Path("../data")
PROCESSED_DIR = DATA_DIR / "processed"
MODEL_DIR = PROCESSED_DIR / "modelling"
ANALYTIC_PATH = PROCESSED_DIR / "crc_analytic_dataset.csv"

MODEL_NAME = "gb_M3_tuned_calibrated"
model, reference_risk_scores, VALID_CATEGORIES, MODEL_REGISTRY = load_prediction_context(MODEL_DIR, ANALYTIC_PATH, MODEL_NAME)
print(f"Loaded '{MODEL_NAME}', plus {len(MODEL_REGISTRY)} models available for cross-model comparison.")
print(f"Reference distribution ready: {len(reference_risk_scores):,} test-set risk scores.")
print(f"Valid categories loaded for {len(VALID_CATEGORIES)} predictors.")

Loaded 'gb_M3_tuned_calibrated', plus 17 models available for cross-model comparison.
Reference distribution ready: 45,530 test-set risk scores.
Valid categories loaded for 17 predictors.


## Valid input values

Every predictor is categorical. `handle_unknown="ignore"` in the underlying encoders means a typo'd category would otherwise be silently zeroed out rather than raising an error -- `predict_respondents` (imported from `pipeline_components.py`, shared with `app.py`, the Streamlit webapp) validates every field against this list first and fails loudly with the valid options, rather than silently mis-scoring a respondent.

## `predict_respondents()`

Defined once in `pipeline_components.py` (shared with `app.py`) so the notebook and the webapp can never drift out of sync. Takes one or more respondents as a list of dicts (every M3 predictor required -- use `"Not reported"` for an unknown value, which every field supports explicitly, rather than omitting the key). Returns the calibrated non-compliance risk, its approximate percentile against the test-set reference distribution, and which risk decile it falls in (Decile 1 = highest risk, matching `5 Evaluation.ipynb`'s risk-decile table).

## Worked examples

Three respondents spanning the range this project's findings would predict: a low-access,
older, insured profile (expected low risk); a young, uninsured, no-personal-doctor profile
(expected high risk, per the odds ratios in `4a` and the permutation importance in `4b`-`4d`);
and a partially-unknown profile using `"Not reported"` for several fields, to confirm the
missingness handling works end to end.

In [2]:
example_respondents = [
    {  # Low risk: older, insured, engaged with care
        "age_group": "70-74", "sex": "Female", "race_ethnicity": "White only, non-Hispanic",
        "education_level": "Graduated college or technical school", "income_group": "$100,000 to < $200,000",
        "employment_status": "Retired", "marital_status": "Married", "urban_rural": "Urban",
        "insurance_status": "Insured", "personal_doctor": "Has personal doctor", "cost_barrier": "No",
        "checkup_recency": "Within past year", "general_health": "Very good", "diabetes_status": "No diabetes",
        "heart_disease": "CHD or MI not reported", "smoking_status": "Never smoked", "bmi_category": "Normal weight",
    },
    {  # High risk: young, uninsured, no access
        "age_group": "45-49", "sex": "Male", "race_ethnicity": "Black only, non-Hispanic",
        "education_level": "Did not graduate high school", "income_group": "Less than $15,000",
        "employment_status": "Unable to work", "marital_status": "Never married", "urban_rural": "Rural",
        "insurance_status": "Uninsured", "personal_doctor": "No personal doctor", "cost_barrier": "Yes",
        "checkup_recency": "5 or more years", "general_health": "Fair", "diabetes_status": "No diabetes",
        "heart_disease": "CHD or MI not reported", "smoking_status": "Current smoker - every day", "bmi_category": "Obese",
    },
    {  # Partially unknown -- several fields "Not reported"
        "age_group": "55-59", "sex": "Female", "race_ethnicity": "Hispanic",
        "education_level": "Not reported", "income_group": "Not reported",
        "employment_status": "Employed for wages", "marital_status": "Married", "urban_rural": "Not reported",
        "insurance_status": "Insured", "personal_doctor": "Has personal doctor", "cost_barrier": "No",
        "checkup_recency": "2 to < 5 years", "general_health": "Good", "diabetes_status": "Not reported",
        "heart_disease": "CHD or MI not reported", "smoking_status": "Former smoker", "bmi_category": "Overweight",
    },
]

results = predict_respondents(model, VALID_CATEGORIES, reference_risk_scores, example_respondents)
results.insert(0, "Profile", ["Low-risk example", "High-risk example", "Partially-unknown example"])
results

,Profile,predicted_risk,percentile_vs_test_set,approx_risk_decile
0,Low-risk example,0.0564,2.7,Decile 10
1,High-risk example,0.9758,99.4,Decile 1 (highest risk)
2,Partially-unknown example,0.4635,82.7,Decile 2


**Reading this:** the low-risk profile scores 5.6% predicted risk (Decile 10, the bottom of the
test-set distribution -- lower risk than 97.3% of test respondents), the high-risk profile scores
97.6% (Decile 1, higher than 99.4% of the test set), and the partially-unknown profile lands at
46.3% (Decile 2, higher than 82.7% of the test set) -- despite having no single strongly
high-risk category, several "Not reported" fields nudge it up considerably, consistent with the
odds-ratio and permutation-importance findings throughout this project that "Not reported" carries
real signal rather than being a neutral placeholder. The percentile/decile context is what makes a
single risk score actionable for outreach -- "97.6%" alone doesn't say much, but "higher risk than
99.4% of the population, comfortably inside the top-decile outreach group" does.

## Model comparison

How much do the different model families agree on this respondent? `predict_across_models`
(also shared with `app.py`) scores the same input through the best-tuned representative of each
family -- logistic regression, random forest, gradient boosting (raw and calibrated), and the
feedforward network -- rather than relying on a single model's number in isolation.

In [3]:
comparison = predict_across_models(MODEL_REGISTRY, VALID_CATEGORIES, example_respondents)
comparison_wide = comparison.pivot(index="respondent_index", columns="model_label", values="predicted_risk")
comparison_wide.insert(0, "Profile", ["Low-risk example", "High-risk example", "Partially-unknown example"])
comparison_wide

model_label,Profile,Feedforward Neural Network (tuned),"Gradient Boosting (tuned, calibrated) -- official model","Gradient Boosting (tuned, raw)",Logistic Regression,Random Forest (tuned)
respondent_index,,,,,,
0,Low-risk example,0.1694,0.0564,0.1686,0.1591,0.1448
1,High-risk example,0.9848,0.9758,0.9699,0.9945,0.9497
2,Partially-unknown example,0.7762,0.4635,0.7089,0.8129,0.7315


**Reading this:** all five models agree on the *order* of the three profiles (low < partially-
unknown < high) -- reassuring, since that's the more fundamental thing to get right for a ranking/
targeting use case. But the absolute numbers disagree more than a quick glance suggests, and the
pattern is systematic, not random: for the low-risk profile, the four uncalibrated models
(logistic regression 15.9%, random forest 14.5%, raw gradient boosting 16.9%, neural network
16.9%) all sit noticeably above the calibrated gradient boosting's 5.6%. For the partially-unknown
profile the gap is much larger -- logistic regression says 81.3%, raw gradient boosting 70.9%,
against the calibrated model's 46.3%.

This is the same `sample_weight="balanced"` effect discussed in `5 Evaluation.ipynb`'s calibration
section, showing up concretely here: **every model except `gb_M3_tuned_calibrated` was trained
with class weighting and never corrected afterward**, so all four are likely inflated above the
true rate, not just the raw gradient boosting model calibration was demonstrated on. The official
model's number is the one that's actually been checked against reality (the reliability diagram
and decile-level predicted-vs-observed comparison in `5`); the other four are useful for
confirming the *ranking* agrees, but their absolute percentages shouldn't be read at face value
the way the calibrated model's can.

## Batch mode: scoring a CSV of new respondents

For scoring many respondents at once: a CSV with one row per respondent and one column per
M3 predictor (same column names and category values as above), no header/id requirements beyond
the 17 predictor columns.

In [4]:
def predict_from_csv(input_csv_path, output_csv_path=None):
    new_respondents = pd.read_csv(input_csv_path, dtype=str)
    predictions = predict_respondents(model, VALID_CATEGORIES, reference_risk_scores, new_respondents.to_dict("records"))
    scored = pd.concat([new_respondents.reset_index(drop=True), predictions], axis=1)
    if output_csv_path is not None:
        scored.to_csv(output_csv_path, index=False)
        print(f"Wrote {len(scored):,} scored respondents to {output_csv_path}")
    return scored

# Example usage (commented out -- point at a real file to use):
# scored = predict_from_csv("../data/new_respondents.csv", "../data/processed/modelling/new_respondents_scored.csv")
print("predict_from_csv(input_csv_path, output_csv_path=None) ready.")

predict_from_csv(input_csv_path, output_csv_path=None) ready.
